# Milestone 11: Measure Performance

This notebook is structured to satisfy all milestone requirements:
1. Implement comprehensive evaluation metrics (accuracy, relevance, coherence, faithfulness).
2. Run human evaluation and A/B comparison across two system variants.
3. Use evaluation tooling (with optional hooks for TruLens or RAGAS).
4. Analyze performance across scenarios, including edge cases and failure modes.

## 0) Setup

If you want to use TruLens or RAGAS directly, uncomment one of the install lines below.

In [1]:
# Optional installs (run once)
# %pip install trulens_eval
# %pip install ragas

from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys

import pandas as pd

## 1) Load Evaluation Dataset

Expected columns in CSV (minimum):
- `id`
- `query`
- `reference_answer`
- `scenario` (normal, edge_case, failure_mode, etc.)

`variant_a_output` and `variant_b_output` are optional if you generate outputs in Section 2 using the app.py models.

Optional human-rating columns:
- `human_relevance_a`, `human_relevance_b` (1-5)
- `human_coherence_a`, `human_coherence_b` (1-5)
- `human_faithfulness_a`, `human_faithfulness_b` (1-5)

In [ ]:
EVAL_PATH = Path('data/modeling_datasets/final/milestone11_eval_set.csv')

if EVAL_PATH.exists():
    eval_df = pd.read_csv(EVAL_PATH)
else:
    eval_df = pd.DataFrame(
        [
            {
                'id': 1,
                'query': (
                    'Who is the best comp for Bryce Young?'
                ),
                'reference_answer': 'A concise comparison with evidence.',
                'scenario': 'normal'
            },
            {
                'id': 2,
                'query': (
                    'Handle missing stats for Bryce Young?'
                ),
                'reference_answer': 'State missing data and provide fallback logic.',
                'scenario': 'edge_case'
            },
            {
                'id': 3,
                'query': (
                    'You are the lead college scouting analyst for Georgia in the 2027 class. Build a full fit '
                    'assessment for Bryce Young against Georgia\'s current QB room and offensive scheme. Include: '
                    'scheme fit strengths and weaknesses, 1-year and 3-year projection, transfer-risk assessment, '
                    'NIL/value justification, and a confidence score from 0 to 100 with explicit uncertainty drivers. '
                    'If key data is missing, state assumptions clearly and show how they change the recommendation.'
                ),
                'reference_answer': 'Comprehensive fit analysis with explicit assumptions, risk, and confidence rationale.',
                'scenario': 'stress_test'
            },
            {
                'id': 4,
                'query': (
                    'You have conflicting inputs: camp reports say elite arm talent, game tape suggests inconsistent '
                    'decision-making, and analytics show strong EPA under pressure but weak red-zone efficiency. '
                    'Produce a weighted evidence table, final grade with rationale, top 3 hypotheses explaining the '
                    'conflict, and what additional data would most reduce uncertainty. End with a go/no-go recommendation '
                    'for an SEC program and defend it.'
                ),
                'reference_answer': 'Reconciled evidence with transparent weighting, hypotheses, and defended recommendation.',
                'scenario': 'conflict_resolution'
            },
            {
                'id': 5,
                'query': (
                    'Evaluate this player for two roles: Role A immediate starter on a rebuilding ACC team; Role B '
                    'developmental backup on a CFP-contending SEC team. For each role, provide expected snap share by year, '
                    'key development milestones, probability of role success, and failure modes with mitigation plan. Then '
                    'decide which role maximizes long-term NFL draft upside and explain tradeoffs.'
                ),
                'reference_answer': 'Role-based comparison with quantified outcomes, risks, and tradeoff-based decision.',
                'scenario': 'multi_scenario'
            },
            {
                'id': 6,
                'query': (
                    'Generate a scouting report when the player has no verified 40-yard dash, incomplete junior-year '
                    'game logs, inconsistent height/weight records across sources, and only two high-quality film samples. '
                    'Quantify data reliability by category, produce a recommendation despite uncertainty, include a fallback '
                    'decision rule if new data arrives, and explicitly flag what could make your recommendation wrong.'
                ),
                'reference_answer': 'Robust recommendation under uncertainty with reliability scoring and failure conditions.',
                'scenario': 'failure_mode'
            }
        ]
    )

eval_df.head(10)

,id,query,reference_answer,scenario
0,1,Who is the best comp for Bryce Young?,A concise comparison with evidence.,normal
1,2,Handle missing stats for Bryce Young?,State missing data and provide fallback logic.,edge_case


In [3]:
# Read model names from app.py without importing it (importing app.py executes Streamlit page code).
import ast

project_root = Path.cwd()
if not (project_root / 'app.py').exists() and (project_root.parent / 'app.py').exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from engine.config import CONFIG as ENGINE_CONFIG
from engine.tools import _get_llm, _llm_response_to_text

def read_app_model_names(app_path: Path) -> tuple[str, str]:
    final_model = ENGINE_CONFIG.get('FINAL_MODEL', 'gemini-3.0-flash')
    summary_model = ENGINE_CONFIG.get('SUMMARY_MODEL', 'gemini-2.5-flash-lite')

    if not app_path.exists():
        return final_model, summary_model

    try:
        tree = ast.parse(app_path.read_text(encoding='utf-8'))
        for node in ast.walk(tree):
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id == 'CONFIG' and isinstance(node.value, ast.Dict):
                        for key_node, val_node in zip(node.value.keys, node.value.values):
                            if isinstance(key_node, ast.Constant) and isinstance(key_node.value, str):
                                key = key_node.value
                                if key == 'FINAL_MODEL' and isinstance(val_node, ast.Constant) and isinstance(val_node.value, str):
                                    final_model = val_node.value
                                if key == 'SUMMARY_MODEL' and isinstance(val_node, ast.Constant) and isinstance(val_node.value, str):
                                    summary_model = val_node.value
    except Exception as exc:
        print(f'Warning: could not parse app.py CONFIG, using engine config defaults. Details: {exc}')

    return final_model, summary_model

FINAL_MODEL_NAME, SUMMARY_MODEL_NAME = read_app_model_names(project_root / 'app.py')

print(f"Variant A model (from app.py CONFIG): {FINAL_MODEL_NAME}")
print(f"Variant B model (from app.py CONFIG): {SUMMARY_MODEL_NAME}")

Variant A model (from app.py CONFIG): gemini-3-flash-preview
Variant B model (from app.py CONFIG): gemini-2.5-flash-lite


## 2) Generate A/B Outputs Using app.py Models

This section reuses the exact model configuration from app.py:
- Variant A: `CONFIG['FINAL_MODEL']`
- Variant B: `CONFIG['SUMMARY_MODEL']`

Set `FORCE_GENERATE_OUTPUTS = True` to overwrite existing outputs.

In [4]:
FORCE_GENERATE_OUTPUTS = False
needs_generation = FORCE_GENERATE_OUTPUTS or not {'variant_a_output', 'variant_b_output'}.issubset(eval_df.columns)

if needs_generation:
    llm_a = _get_llm(FINAL_MODEL_NAME, temperature=0.0, max_output_tokens=450)
    llm_b = _get_llm(SUMMARY_MODEL_NAME, temperature=0.0, max_output_tokens=450)

    if llm_a is None or llm_b is None:
        raise RuntimeError(
            'Could not initialize LLM. Ensure GEMINI_API_KEY and langchain-google-genai are available.'
        )

    def build_eval_prompt(query: str) -> str:
        return (
            'You are a football scouting assistant. Answer the query in 3-5 sentences with concise evidence. '
            'If data is missing, explicitly state the gap and provide a fallback recommendation.\n\n'
            f'Query: {query}'
        )

    def run_variant(llm, query: str) -> str:
        response = llm.invoke(build_eval_prompt(query))
        return _llm_response_to_text(response).strip()

    eval_df['variant_a_output'] = eval_df['query'].apply(lambda q: run_variant(llm_a, q))
    eval_df['variant_b_output'] = eval_df['query'].apply(lambda q: run_variant(llm_b, q))

eval_df[['id', 'query', 'variant_a_output', 'variant_b_output']].head()

,id,query,variant_a_output,variant_b_output
0,1,Who is the best comp for Bryce Young?,The most accurate stylistic comparison for Bry...,"Bryce Young's exceptional pocket presence, qui..."
1,2,Handle missing stats for Bryce Young?,To address missing granular data for Bryce You...,Bryce Young's collegiate statistics are readil...


## 3) Metric Definitions

This includes:
- Accuracy (exact-match style against reference)
- Relevance (token overlap proxy and optional human score)
- Coherence (length and sentence-structure proxy plus optional human score)
- Faithfulness (reference overlap proxy plus optional human score)

In [6]:
def normalize_text(text: str) -> str:
    return ' '.join(str(text).strip().lower().split())


def exact_match(pred: str, ref: str) -> int:
    return int(normalize_text(pred) == normalize_text(ref))


def token_set(text: str) -> set[str]:
    return set(normalize_text(text).split())


def jaccard_similarity(a: str, b: str) -> float:
    a_set = token_set(a)
    b_set = token_set(b)
    union = a_set | b_set
    if not union:
        return 1.0
    return len(a_set & b_set) / len(union)


def coherence_proxy(text: str) -> float:
    tokens = normalize_text(text).split()
    if not tokens:
        return 0.0
    sentence_count = max(1, str(text).count('.') + str(text).count('!') + str(text).count('?'))
    avg_sentence_len = len(tokens) / sentence_count
    # Reward a moderate sentence length range as a simple coherence heuristic.
    return float(max(0.0, 1.0 - abs(avg_sentence_len - 18.0) / 25.0))


def compute_variant_metrics(df: pd.DataFrame, variant: str) -> pd.DataFrame:
    output_col = f'variant_{variant}_output'
    rel_col = f'human_relevance_{variant}'
    coh_col = f'human_coherence_{variant}'
    fai_col = f'human_faithfulness_{variant}'

    metrics_df = df.copy()
    metrics_df[f'accuracy_{variant}'] = metrics_df.apply(
        lambda r: exact_match(r[output_col], r['reference_answer']), axis=1
    )
    metrics_df[f'relevance_proxy_{variant}'] = metrics_df.apply(
        lambda r: jaccard_similarity(r[output_col], r['query']), axis=1
    )
    metrics_df[f'faithfulness_proxy_{variant}'] = metrics_df.apply(
        lambda r: jaccard_similarity(r[output_col], r['reference_answer']), axis=1
    )
    metrics_df[f'coherence_proxy_{variant}'] = metrics_df[output_col].apply(coherence_proxy)

    if rel_col in metrics_df.columns:
        metrics_df[f'relevance_{variant}'] = metrics_df[rel_col] / 5.0
    else:
        metrics_df[f'relevance_{variant}'] = metrics_df[f'relevance_proxy_{variant}']

    if coh_col in metrics_df.columns:
        metrics_df[f'coherence_{variant}'] = metrics_df[coh_col] / 5.0
    else:
        metrics_df[f'coherence_{variant}'] = metrics_df[f'coherence_proxy_{variant}']

    if fai_col in metrics_df.columns:
        metrics_df[f'faithfulness_{variant}'] = metrics_df[fai_col] / 5.0
    else:
        metrics_df[f'faithfulness_{variant}'] = metrics_df[f'faithfulness_proxy_{variant}']

    metrics_df[f'composite_{variant}'] = (
        0.35 * metrics_df[f'accuracy_{variant}']
        + 0.25 * metrics_df[f'relevance_{variant}']
        + 0.20 * metrics_df[f'coherence_{variant}']
        + 0.20 * metrics_df[f'faithfulness_{variant}']
    )

    return metrics_df

## 4) Compute Metrics for A/B Variants

In [7]:
metrics_a = compute_variant_metrics(eval_df, 'a')
metrics_ab = compute_variant_metrics(metrics_a, 'b')

summary = pd.DataFrame(
    {
        'metric': ['accuracy', 'relevance', 'coherence', 'faithfulness', 'composite'],
        'variant_a': [
            metrics_ab['accuracy_a'].mean(),
            metrics_ab['relevance_a'].mean(),
            metrics_ab['coherence_a'].mean(),
            metrics_ab['faithfulness_a'].mean(),
            metrics_ab['composite_a'].mean(),
        ],
        'variant_b': [
            metrics_ab['accuracy_b'].mean(),
            metrics_ab['relevance_b'].mean(),
            metrics_ab['coherence_b'].mean(),
            metrics_ab['faithfulness_b'].mean(),
            metrics_ab['composite_b'].mean(),
        ],
    }
)
summary['delta_a_minus_b'] = summary['variant_a'] - summary['variant_b']
summary

,metric,variant_a,variant_b,delta_a_minus_b
0,accuracy,0.000000,0.000000,0.000000
1,relevance,0.190058,0.052529,0.137530
2,coherence,0.880000,0.898333,-0.018333
3,faithfulness,0.106443,0.028745,0.077698
4,composite,0.244803,0.198548,0.046255


## 5) Scenario Analysis (Edge Cases and Failure Modes)

In [9]:
scenario_summary = (
    metrics_ab.groupby('scenario', dropna=False)[
        [
            'accuracy_a', 'accuracy_b',
            'relevance_a', 'relevance_b',
            'coherence_a', 'coherence_b',
            'faithfulness_a', 'faithfulness_b',
            'composite_a', 'composite_b',
        ]
    ]
    .mean()
    .sort_index()
)
scenario_summary

,accuracy_a,accuracy_b,relevance_a,relevance_b,coherence_a,coherence_b,faithfulness_a,faithfulness_b,composite_a,composite_b
scenario,,,,,,,,,,
edge_case,0.0,0.0,0.157895,0.051724,0.92,0.906667,0.095238,0.016393,0.242521,0.197543
normal,0.0,0.0,0.222222,0.053333,0.84,0.890000,0.117647,0.041096,0.247085,0.199553


## 6) Optional: TruLens / RAGAS Integration Notes

- TruLens: wrap your generation call and log feedback functions for relevance/groundedness.
- RAGAS: create a dataset with `question`, `answer`, `contexts`, `ground_truth` and compute standard RAG metrics.

Use these tools if your instructor expects named frameworks in addition to custom metrics.

In [10]:
OUTPUT_DIR = Path('data/modeling_datasets/final')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_path = OUTPUT_DIR / f'milestone11_summary_{run_id}.csv'
scenario_path = OUTPUT_DIR / f'milestone11_scenarios_{run_id}.csv'

summary.to_csv(summary_path, index=False)
scenario_summary.to_csv(scenario_path)

print(f'Saved summary to: {summary_path}')
print(f'Saved scenario analysis to: {scenario_path}')

Saved summary to: data\modeling_datasets\final\milestone11_summary_20260330_220821.csv
Saved scenario analysis to: data\modeling_datasets\final\milestone11_scenarios_20260330_220821.csv
